# 🐷 K-Fold Ensembling Inference

**Das Herzstück des Setups!** 
Dieses Skript lädt ALLE 5 Modelle, die das Notebook `kfold_train.ipynb` erzeugt hat. Es lässt alle 5 Modelle ihre individuellen Vorhersagen für jedes Bild treffen.
Anschließend werden die Ergebnisse gemittelt (Ensembling). Fehler eines Modells werden durch korrekte Schätzungen der restlichen vier eliminiert.

In [2]:
import os, ast
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm
import warnings; warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TAG = "T2"
DATA_ROOT = "/datasets/multi-view-pig-posture-recognition"
TEST_CSV  = f"{DATA_ROOT}/test.csv"
IMG_DIR   = f"{DATA_ROOT}/test_images"

# Wir lesen alle Checkpoints ein, die das Wort 'fold' enthalten
CKPT_DIR  = f"runs/kfold_{TAG.lower()}"
CKPT_PATHS = [os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR) if f.startswith("best_model_fold_")]

OUTPUT_FILE = f"{TAG}_kfold_ensemble_submission.csv"
IMG_SIZE    = 288
BATCH_SIZE  = 64
NUM_WORKERS = 8
USE_TTA     = True   # Multipliziert mit 5 Modellen = extrem stark (20 Predictions pro Bild!)
PAD_RATIO   = 0.25
NUM_CLASSES = 5

print(f"Einlese-Modelle ({len(CKPT_PATHS)} gefunden): \n" + "\n".join(CKPT_PATHS))

Einlese-Modelle (5 gefunden): 
runs/kfold_t2/best_model_fold_5.pth
runs/kfold_t2/best_model_fold_1.pth
runs/kfold_t2/best_model_fold_3.pth
runs/kfold_t2/best_model_fold_4.pth
runs/kfold_t2/best_model_fold_2.pth


In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, row["row_id"]

S = IMG_SIZE
NORM = [[0.485,0.456,0.406],[0.229,0.224,0.225]]
TTA_TRANSFORMS = [
    T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
]

In [4]:
test_df = pd.read_csv(TEST_CSV)

@torch.no_grad()
def predict_tta(model, df, img_dir, transforms):
    all_probs = []
    for i, tf in enumerate(transforms):
        ds     = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"    TTA {i+1}/{len(transforms)}", leave=False):
            with autocast(): logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

transforms = TTA_TRANSFORMS if USE_TTA else [TTA_TRANSFORMS[0]]
ensemble_probs = []

for fold, path in enumerate(CKPT_PATHS):
    print(f"\n🤖 Modell {fold+1}/{len(CKPT_PATHS)} wird geladen ({os.path.basename(path)}):")
    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "convnext_base")
    model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE).eval()
    
    fold_prob = predict_tta(model, test_df, IMG_DIR, transforms)
    ensemble_probs.append(fold_prob)
    del model; torch.cuda.empty_cache()

predictions = np.mean(ensemble_probs, axis=0).argmax(axis=1)
print(f"\n✓ Vorhersagen für {len(predictions)} Instanzen basierend auf {len(CKPT_PATHS)} Modellen abgeschlossen.")


🤖 Modell 1/5 wird geladen (best_model_fold_5.pth):


    TTA 1/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 2/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 3/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 4/4:   0%|          | 0/183 [00:00<?, ?it/s]

...

    TTA 1/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 2/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 3/4:   0%|          | 0/183 [00:00<?, ?it/s]

    TTA 4/4:   0%|          | 0/183 [00:00<?, ?it/s]


✓ Vorhersagen für 11708 Instanzen basierend auf 5 Modellen abgeschlossen.


In [5]:
submission = pd.DataFrame({"row_id": test_df["row_id"].values, "class_id": predictions.astype(int)})
submission.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Ensembled Submission gespeichert -> {OUTPUT_FILE}  ({len(submission)} Reihen)")
submission.head(10)

✓ Ensembled Submission gespeichert -> T2_kfold_ensemble_submission.csv  (11708 Reihen)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,1
2,test_pen1_tur_cam1_20250920_174649_0002,0
3,test_pen1_tur_cam1_20250920_174649_0003,1
4,test_pen1_tur_cam1_20250920_174649_0004,1
5,test_pen1_tur_cam1_20250920_174649_0005,1
6,test_pen1_tur_cam1_20250920_174649_0006,0
7,test_pen1_tur_cam1_20250920_174649_0007,0
8,test_pen1_tur_cam1_20250920_174649_0008,3
9,test_pen1_tur_cam1_20250921_050022_0000,1
